# Task 1 — Post-Incident Traffic Forecasting

**Goal:** implement TraffiDent-style Task 1:

> Predict traffic flow following accident/incident for the next **1, 3, and 6 hours** given **24 hours of historical observations**.

This notebook uses:

- **Dataset:** `final_hourly_flow_allfeature.csv`
- **Target:** `total_flow`
- **Input length:** 24 historical hours
- **Forecast horizons:** `t+1`, `t+3`, `t+6`
- **Baselines:**
  1. Historical Average (HA)
  2. Linear Regression (LR)

## Fixes included in this version

1. Uses **time-based split**, not random split.
2. Removes daytime zero-flow anomalies using `is_anomaly == 0`.
3. Uses **Masked MAPE** to avoid inflated MAPE from near-zero traffic flow.
4. Defines **Incident Test** as post-incident/incident-affected samples.
5. Prevents future leakage by using only features available at or before anchor time `t`.

In [1]:
# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Change this path if your CSV is in another folder.
# Recommended: put this notebook in the same folder as final_hourly_flow_allfeature.csv.
DATA_PATH = "final_hourly_flow_allfeature.csv"

# Alternative example for Mac absolute path:
DATA_PATH = "/Users/andishahifahmuthahharah/Downloads/Dataset_main/Data/final_hourly_flow_allfeature.csv"

INPUT_HOURS = 24
HORIZONS = [1, 3, 6]

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20

# For Masked MAPE: ignore actual flow <= threshold.
MAPE_THRESHOLD = 10.0

RANDOM_STATE = 42

In [2]:
# ============================================================
# 1. Load data
# ============================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}\n\n"
        "Solutions:\n"
        "1. Put final_hourly_flow_allfeature.csv in the same folder as this notebook and use:\n"
        "   DATA_PATH = 'final_hourly_flow_allfeature.csv'\n\n"
        "or\n"
        "2. Use the full absolute path, for example on Mac:\n"
        "   DATA_PATH = '/Users/your_username/Downloads/Dataset_main/Data/final_hourly_flow_allfeature.csv'"
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Raw shape:", df.shape)
display(df.head())
display(df.dtypes)

Raw shape: (1007400, 24)


,station_id,timestamp,total_flow,wgs84_latitude,wgs84_longitude,road_name,suburb,post_code,device_type,quality_rating,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation,weather_code,apparent_temperature,temperature_2m,wind_gusts_10m,relative_humidity,incident_count,is_anomaly
0,100001,2025-01-01 00:00:00,13.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
1,100001,2025-01-01 01:00:00,9.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
2,100001,2025-01-01 02:00:00,10.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.952227,21.00,5.040000,86.98510,0,0
3,100001,2025-01-01 03:00:00,6.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.543003,20.65,5.760000,88.87851,0,0
4,100001,2025-01-01 04:00:00,7.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.117048,20.25,6.479999,90.81496,0,0


station_id                    object
timestamp                     object
total_flow                   float64
wgs84_latitude               float64
wgs84_longitude              float64
road_name                     object
suburb                        object
post_code                      int64
device_type                   object
quality_rating                 int64
lane_count                     int64
road_functional_hierarchy      int64
distance_to_intersection     float64
incident_id                   object
is_major_incident              int64
impact_sequence_hour         float64
precipitation                float64
weather_code                 float64
apparent_temperature         float64
temperature_2m               float64
wind_gusts_10m               float64
relative_humidity            float64
incident_count                 int64
is_anomaly                     int64
dtype: object

In [3]:
# ============================================================
# 2. Basic cleaning
# ============================================================

required_cols = ["station_id", "timestamp", "total_flow"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df = df.copy()

df["station_id"] = (
    df["station_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["total_flow"] = pd.to_numeric(df["total_flow"], errors="coerce")

df = df.dropna(subset=["station_id", "timestamp", "total_flow"]).copy()
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print("After basic cleaning:", df.shape)
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Number of stations:", df["station_id"].nunique())
display(df.head())

After basic cleaning: (1007400, 24)
Date range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
Number of stations: 115


,station_id,timestamp,total_flow,wgs84_latitude,wgs84_longitude,road_name,suburb,post_code,device_type,quality_rating,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation,weather_code,apparent_temperature,temperature_2m,wind_gusts_10m,relative_humidity,incident_count,is_anomaly
0,100001,2025-01-01 00:00:00,13.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
1,100001,2025-01-01 01:00:00,9.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
2,100001,2025-01-01 02:00:00,10.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.952227,21.00,5.040000,86.98510,0,0
3,100001,2025-01-01 03:00:00,6.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.543003,20.65,5.760000,88.87851,0,0
4,100001,2025-01-01 04:00:00,7.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.117048,20.25,6.479999,90.81496,0,0


In [4]:
# ============================================================
# 3. Remove daytime zero-flow anomalies
# ============================================================

# is_anomaly = 1 means daytime zero-flow anomaly, likely sensor failure.
# We remove it from training and evaluation.

if "is_anomaly" in df.columns:
    before = len(df)
    df["is_anomaly"] = pd.to_numeric(df["is_anomaly"], errors="coerce").fillna(0).astype(int)
    df = df[df["is_anomaly"] == 0].copy()
    after = len(df)
    print(f"Removed anomaly rows: {before - after:,}")
else:
    print("Column is_anomaly not found. No anomaly filtering applied.")

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
print("After anomaly filtering:", df.shape)

Removed anomaly rows: 162,810
After anomaly filtering: (844590, 24)


In [5]:
# ============================================================
# 4. Create incident/post-incident flags
# ============================================================

# Normal state:
# - incident_count = 0
# - impact_sequence_hour = -1
#
# Incident/post-incident affected state:
# - incident_count > 0 OR impact_sequence_hour >= 0

if "incident_count" not in df.columns:
    df["incident_count"] = 0

if "impact_sequence_hour" not in df.columns:
    df["impact_sequence_hour"] = -1

if "is_major_incident" not in df.columns:
    df["is_major_incident"] = 0

df["incident_count"] = pd.to_numeric(df["incident_count"], errors="coerce").fillna(0)
df["impact_sequence_hour"] = pd.to_numeric(df["impact_sequence_hour"], errors="coerce").fillna(-1)
df["is_major_incident"] = pd.to_numeric(df["is_major_incident"], errors="coerce").fillna(0)

df["is_incident_or_post"] = (
    (df["incident_count"] > 0) |
    (df["impact_sequence_hour"] >= 0)
).astype(int)

print("Incident/post-incident row count:", int(df["is_incident_or_post"].sum()))
print("Normal row count:", int((df["is_incident_or_post"] == 0).sum()))
display(df[["station_id", "timestamp", "total_flow", "incident_count", "impact_sequence_hour", "is_incident_or_post"]].head())

Incident/post-incident row count: 7571
Normal row count: 837019


,station_id,timestamp,total_flow,incident_count,impact_sequence_hour,is_incident_or_post
0,100001,2025-01-01 00:00:00,13.0,0,-1.0,0
1,100001,2025-01-01 01:00:00,9.0,0,-1.0,0
2,100001,2025-01-01 02:00:00,10.0,0,-1.0,0
3,100001,2025-01-01 03:00:00,6.0,0,-1.0,0
4,100001,2025-01-01 04:00:00,7.0,0,-1.0,0


In [6]:
# ============================================================
# 5. Feature engineering: temporal features and lag features
# ============================================================

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# Calendar/time features available at anchor time t.
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# Cyclical encoding.
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

# 24-hour historical observations:
# lag_1 = total_flow at t-1, ..., lag_24 = total_flow at t-24
for lag in range(1, INPUT_HOURS + 1):
    df[f"flow_lag_{lag}"] = df.groupby("station_id")["total_flow"].shift(lag)

lag_cols = [f"flow_lag_{lag}" for lag in range(1, INPUT_HOURS + 1)]

# Historical Average baseline using the 24 previous hours only.
df["pred_HA"] = df[lag_cols].mean(axis=1)

# Future targets.
for h in HORIZONS:
    df[f"y_t_plus_{h}"] = df.groupby("station_id")["total_flow"].shift(-h)

print("Feature engineering complete.")
display(df[["station_id", "timestamp", "total_flow", "pred_HA"] + lag_cols[:5] + [f"y_t_plus_{h}" for h in HORIZONS]].head(30))

Feature engineering complete.


,station_id,timestamp,total_flow,pred_HA,flow_lag_1,flow_lag_2,flow_lag_3,flow_lag_4,flow_lag_5,y_t_plus_1,y_t_plus_3,y_t_plus_6
0,100001,2025-01-01 00:00:00,13.0,NaN,NaN,NaN,NaN,NaN,NaN,9.0,6.0,14.0
1,100001,2025-01-01 01:00:00,9.0,13.000000,13.0,NaN,NaN,NaN,NaN,10.0,7.0,11.0
2,100001,2025-01-01 02:00:00,10.0,11.000000,9.0,13.0,NaN,NaN,NaN,6.0,8.0,12.0
3,100001,2025-01-01 03:00:00,6.0,10.666667,10.0,9.0,13.0,NaN,NaN,7.0,14.0,15.0
4,100001,2025-01-01 04:00:00,7.0,9.500000,6.0,10.0,9.0,13.0,NaN,8.0,11.0,11.0
5,100001,2025-01-01 05:00:00,8.0,9.000000,7.0,6.0,10.0,9.0,13.0,14.0,12.0,16.0
6,100001,2025-01-01 06:00:00,14.0,8.833333,8.0,7.0,6.0,10.0,9.0,11.0,15.0,23.0
7,100001,2025-01-01 07:00:00,11.0,9.571429,14.0,8.0,7.0,6.0,10.0,12.0,11.0,23.0
8,100001,2025-01-01 08:00:00,12.0,9.750000,11.0,14.0,8.0,7.0,6.0,15.0,16.0,20.0
9,100001,2025-01-01 09:00:00,15.0,10.000000,12.0,11.0,14.0,8.0,7.0,11.0,23.0,20.0


In [7]:
# ============================================================
# 6. Build modeling samples
# ============================================================

target_cols = [f"y_t_plus_{h}" for h in HORIZONS]
sample_required = lag_cols + ["pred_HA"] + target_cols

samples = df.dropna(subset=sample_required).copy()

# Incident anchor: anchor t is currently incident/post-incident affected.
samples["is_incident_anchor"] = samples["is_incident_or_post"].astype(int)

print("Samples shape:", samples.shape)
print("Incident anchor samples:", int(samples["is_incident_anchor"].sum()))
print("General samples:", len(samples))
display(samples.head())

Samples shape: (841140, 62)
Incident anchor samples: 7542
General samples: 841140


,station_id,timestamp,total_flow,wgs84_latitude,wgs84_longitude,road_name,suburb,post_code,device_type,quality_rating,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation,weather_code,apparent_temperature,temperature_2m,wind_gusts_10m,relative_humidity,incident_count,is_anomaly,is_incident_or_post,hour,dayofweek,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,flow_lag_1,flow_lag_2,flow_lag_3,flow_lag_4,flow_lag_5,flow_lag_6,flow_lag_7,flow_lag_8,flow_lag_9,flow_lag_10,flow_lag_11,flow_lag_12,flow_lag_13,flow_lag_14,flow_lag_15,flow_lag_16,flow_lag_17,flow_lag_18,flow_lag_19,flow_lag_20,flow_lag_21,flow_lag_22,flow_lag_23,flow_lag_24,pred_HA,y_t_plus_1,y_t_plus_3,y_t_plus_6,is_incident_anchor
24,100001,2025-01-02 00:00:00,17.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,23.729820,22.50,34.200000,81.120926,0,0,0,0,3,1,0,0.000000,1.000000,0.433884,-0.900969,10.0,17.0,16.0,14.0,20.0,14.0,20.0,22.0,20.0,20.0,23.0,23.0,16.0,11.0,15.0,12.0,11.0,14.0,8.0,7.0,6.0,10.0,9.0,13.0,14.625000,11.0,18.0,101.0,0
25,100001,2025-01-02 01:00:00,11.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,2.0,22.799866,21.75,36.719997,84.913445,0,0,0,1,3,1,0,0.258819,0.965926,0.433884,-0.900969,17.0,10.0,17.0,16.0,14.0,20.0,14.0,20.0,22.0,20.0,20.0,23.0,23.0,16.0,11.0,15.0,12.0,11.0,14.0,8.0,7.0,6.0,10.0,9.0,14.791667,15.0,33.0,103.0,0
26,100001,2025-01-02 02:00:00,15.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,3.0,21.972277,21.25,39.600000,86.466820,0,0,0,2,3,1,0,0.500000,0.866025,0.433884,-0.900969,11.0,17.0,10.0,17.0,16.0,14.0,20.0,14.0,20.0,22.0,20.0,20.0,23.0,23.0,16.0,11.0,15.0,12.0,11.0,14.0,8.0,7.0,6.0,10.0,14.875000,18.0,67.0,90.0,0
27,100001,2025-01-02 03:00:00,18.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,3.0,21.072800,20.80,43.199997,85.346850,0,0,0,3,3,1,0,0.707107,0.707107,0.433884,-0.900969,15.0,11.0,17.0,10.0,17.0,16.0,14.0,20.0,14.0,20.0,22.0,20.0,20.0,23.0,23.0,16.0,11.0,15.0,12.0,11.0,14.0,8.0,7.0,6.0,15.083333,33.0,101.0,107.0,0
28,100001,2025-01-02 04:00:00,33.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,3.0,21.074910,20.55,42.120000,84.254250,0,0,0,4,3,1,0,0.866025,0.500000,0.433884,-0.900969,18.0,15.0,11.0,17.0,10.0,17.0,16.0,14.0,20.0,14.0,20.0,22.0,20.0,20.0,23.0,23.0,16.0,11.0,15.0,12.0,11.0,14.0,8.0,7.0,15.583333,67.0,103.0,94.0,0


In [8]:
# ============================================================
# 7. Time-based train / validation / test split
# ============================================================

# Split by unique timestamps to avoid temporal leakage.
unique_times = np.array(sorted(samples["timestamp"].unique()))
n_times = len(unique_times)

train_end_idx = int(n_times * TRAIN_RATIO)
val_end_idx = int(n_times * (TRAIN_RATIO + VAL_RATIO))

train_end_time = unique_times[train_end_idx - 1]
val_end_time = unique_times[val_end_idx - 1]

samples["split"] = np.where(
    samples["timestamp"] <= train_end_time,
    "train",
    np.where(samples["timestamp"] <= val_end_time, "val", "test")
)

print("Train end:", train_end_time)
print("Validation end:", val_end_time)
print(samples["split"].value_counts())

print("\nIncident samples per split:")
display(pd.crosstab(samples["split"], samples["is_incident_anchor"]))

display(samples[["station_id", "timestamp", "split", "is_incident_anchor", "total_flow"]].head())

Train end: 2025-09-18 03:00:00
Validation end: 2025-10-22 23:00:00
split
train    589584
test     165204
val       86352
Name: count, dtype: int64

Incident samples per split:


is_incident_anchor,0,1
split,,
test,164278,926
train,583818,5766
val,85502,850


,station_id,timestamp,split,is_incident_anchor,total_flow
24,100001,2025-01-02 00:00:00,train,0,17.0
25,100001,2025-01-02 01:00:00,train,0,11.0
26,100001,2025-01-02 02:00:00,train,0,15.0
27,100001,2025-01-02 03:00:00,train,0,18.0
28,100001,2025-01-02 04:00:00,train,0,33.0


In [9]:
# ============================================================
# 8. Define Linear Regression features
# ============================================================

# Use only features available at anchor time t or before.
# Do NOT use future target variables or future incident/weather values.

numeric_candidate_features = []

# 24 historical flow values.
numeric_candidate_features += lag_cols

# Calendar features.
numeric_candidate_features += [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "is_weekend",
    "month",
]

# Current-time incident features.
numeric_candidate_features += [
    "incident_count",
    "is_major_incident",
    "impact_sequence_hour",
    "is_incident_or_post",
]

# Weather at anchor time t, if available.
numeric_candidate_features += [
    "precipitation",
    "weather_code",
    "apparent_temperature",
    "temperature_2m",
    "wind_gusts_10m",
    "relative_humidity",
]

# Static road features, if available.
numeric_candidate_features += [
    "wgs84_latitude",
    "wgs84_longitude",
    "lane_count",
    "road_functional_hierarchy",
    "distance_to_intersection",
    "quality_rating",
]

numeric_features = [c for c in numeric_candidate_features if c in samples.columns]

for c in numeric_features:
    samples[c] = pd.to_numeric(samples[c], errors="coerce")

categorical_candidate_features = [
    "station_id",
    "road_name",
    "suburb",
    "post_code",
    "device_type",
]
categorical_features = [c for c in categorical_candidate_features if c in samples.columns]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Numeric features: ['flow_lag_1', 'flow_lag_2', 'flow_lag_3', 'flow_lag_4', 'flow_lag_5', 'flow_lag_6', 'flow_lag_7', 'flow_lag_8', 'flow_lag_9', 'flow_lag_10', 'flow_lag_11', 'flow_lag_12', 'flow_lag_13', 'flow_lag_14', 'flow_lag_15', 'flow_lag_16', 'flow_lag_17', 'flow_lag_18', 'flow_lag_19', 'flow_lag_20', 'flow_lag_21', 'flow_lag_22', 'flow_lag_23', 'flow_lag_24', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'month', 'incident_count', 'is_major_incident', 'impact_sequence_hour', 'is_incident_or_post', 'precipitation', 'weather_code', 'apparent_temperature', 'temperature_2m', 'wind_gusts_10m', 'relative_humidity', 'wgs84_latitude', 'wgs84_longitude', 'lane_count', 'road_functional_hierarchy', 'distance_to_intersection', 'quality_rating']
Categorical features: ['station_id', 'road_name', 'suburb', 'post_code', 'device_type']
Number of numeric features: 46
Number of categorical features: 5


In [10]:
# ============================================================
# 9. Metrics
# ============================================================

def masked_mape(y_true, y_pred, threshold=MAPE_THRESHOLD):
    """
    MAPE can explode when actual values are zero or near-zero.
    This version ignores rows where y_true <= threshold.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > threshold
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "MAE": np.nan,
            "RMSE": np.nan,
            "Masked_MAPE": np.nan,
            "N": 0,
        }

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "Masked_MAPE": masked_mape(y_true, y_pred),
        "N": len(y_true),
    }

In [11]:
# ============================================================
# 10. Train Linear Regression models for t+1, t+3, t+6
# ============================================================

train_data = samples[samples["split"] == "train"].copy()
val_data = samples[samples["split"] == "val"].copy()
test_data = samples[samples["split"] == "test"].copy()

X_train = train_data[numeric_features + categorical_features]
X_val = val_data[numeric_features + categorical_features]
X_test = test_data[numeric_features + categorical_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

lr_models = {}

for h in HORIZONS:
    target_col = f"y_t_plus_{h}"
    y_train = train_data[target_col]

    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ])

    model.fit(X_train, y_train)
    lr_models[h] = model

    samples.loc[samples["split"] == "train", f"pred_LR_t_plus_{h}"] = model.predict(X_train)
    samples.loc[samples["split"] == "val", f"pred_LR_t_plus_{h}"] = model.predict(X_val)
    samples.loc[samples["split"] == "test", f"pred_LR_t_plus_{h}"] = model.predict(X_test)

    print(f"Trained Linear Regression for horizon t+{h}.")

Trained Linear Regression for horizon t+1.
Trained Linear Regression for horizon t+3.
Trained Linear Regression for horizon t+6.


In [12]:
# ============================================================
# 11. Evaluation: General Test vs Incident/Post-Incident Test
# ============================================================

results = []

eval_sets = {
    "General Test": samples["split"].eq("test"),
    "Incident/Post-Incident Test": samples["split"].eq("test") & samples["is_incident_anchor"].eq(1),
}

for eval_name, mask in eval_sets.items():
    subset = samples.loc[mask].copy()

    if subset.empty:
        print(f"Warning: {eval_name} has 0 samples.")
        continue

    for h in HORIZONS:
        target_col = f"y_t_plus_{h}"

        metric = evaluate_predictions(subset[target_col], subset["pred_HA"])
        metric.update({
            "Eval_Set": eval_name,
            "Model": "Historical Average",
            "Horizon": f"t+{h}",
        })
        results.append(metric)

        pred_col = f"pred_LR_t_plus_{h}"
        metric = evaluate_predictions(subset[target_col], subset[pred_col])
        metric.update({
            "Eval_Set": eval_name,
            "Model": "Linear Regression",
            "Horizon": f"t+{h}",
        })
        results.append(metric)

results_df = pd.DataFrame(results)
results_df = results_df[["Eval_Set", "Model", "Horizon", "MAE", "RMSE", "Masked_MAPE", "N"]]

display(results_df)

,Eval_Set,Model,Horizon,MAE,RMSE,Masked_MAPE,N
0,General Test,Historical Average,t+1,239.392180,436.125259,256.138015,165204
1,General Test,Linear Regression,t+1,99.234342,190.738497,95.439052,165204
2,General Test,Historical Average,t+3,254.429298,458.931230,272.667795,165204
3,General Test,Linear Regression,t+3,168.092454,286.553609,177.803069,165204
4,General Test,Historical Average,t+6,272.777902,484.361183,296.537401,165204
5,General Test,Linear Regression,t+6,231.489461,358.393577,266.101385,165204
6,Incident/Post-Incident Test,Historical Average,t+1,373.882919,584.706411,404.690660,926
7,Incident/Post-Incident Test,Linear Regression,t+1,160.813329,286.225754,102.816911,926
8,Incident/Post-Incident Test,Historical Average,t+3,405.753060,630.482040,398.806150,926
9,Incident/Post-Incident Test,Linear Regression,t+3,265.596990,415.070097,204.634646,926


In [13]:
# ============================================================
# 12. Neat result table
# ============================================================

pivot_results = results_df.pivot_table(
    index=["Eval_Set", "Model"],
    columns="Horizon",
    values=["MAE", "RMSE", "Masked_MAPE"],
    aggfunc="first"
)

display(pivot_results)

MAE                         Masked_MAPE                                RMSE                        
Horizon                                                t+1         t+3         t+6         t+1         t+3         t+6         t+1         t+3         t+6
Eval_Set                    Model                                                                                                                         
General Test                Historical Average  239.392180  254.429298  272.777902  256.138015  272.667795  296.537401  436.125259  458.931230  484.361183
                            Linear Regression    99.234342  168.092454  231.489461   95.439052  177.803069  266.101385  190.738497  286.553609  358.393577
Incident/Post-Incident Test Historical Average  373.882919  405.753060  429.394708  404.690660  398.806150  414.986481  584.706411  630.482040  659.847675
                            Linear Regression   160.813329  265.596990  343.450874  102.816911  204.634646  309.607213  286.225754  415.070097  477.262788

In [14]:
# ============================================================
# 13. Compare General vs Incident Test difficulty
# ============================================================

comparison_rows = []

for model_name in results_df["Model"].unique():
    for h in [f"t+{x}" for x in HORIZONS]:
        g = results_df[
            (results_df["Eval_Set"] == "General Test") &
            (results_df["Model"] == model_name) &
            (results_df["Horizon"] == h)
        ]
        i = results_df[
            (results_df["Eval_Set"] == "Incident/Post-Incident Test") &
            (results_df["Model"] == model_name) &
            (results_df["Horizon"] == h)
        ]

        if len(g) == 0 or len(i) == 0:
            continue

        g_mae = float(g["MAE"].iloc[0])
        i_mae = float(i["MAE"].iloc[0])
        g_rmse = float(g["RMSE"].iloc[0])
        i_rmse = float(i["RMSE"].iloc[0])

        comparison_rows.append({
            "Model": model_name,
            "Horizon": h,
            "General_MAE": g_mae,
            "Incident_MAE": i_mae,
            "MAE_Increase": i_mae - g_mae,
            "MAE_Increase_%": ((i_mae - g_mae) / g_mae) * 100 if g_mae != 0 else np.nan,
            "General_RMSE": g_rmse,
            "Incident_RMSE": i_rmse,
            "RMSE_Increase": i_rmse - g_rmse,
            "RMSE_Increase_%": ((i_rmse - g_rmse) / g_rmse) * 100 if g_rmse != 0 else np.nan,
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,Model,Horizon,General_MAE,Incident_MAE,MAE_Increase,MAE_Increase_%,General_RMSE,Incident_RMSE,RMSE_Increase,RMSE_Increase_%
0,Historical Average,t+1,239.392180,373.882919,134.490739,56.180089,436.125259,584.706411,148.581152,34.068458
1,Historical Average,t+3,254.429298,405.753060,151.323762,59.475762,458.931230,630.482040,171.550810,37.380505
2,Historical Average,t+6,272.777902,429.394708,156.616806,57.415503,484.361183,659.847675,175.486492,36.230503
3,Linear Regression,t+1,99.234342,160.813329,61.578987,62.054109,190.738497,286.225754,95.487257,50.061869
4,Linear Regression,t+3,168.092454,265.596990,97.504535,58.006492,286.553609,415.070097,128.516488,44.849021
5,Linear Regression,t+6,231.489461,343.450874,111.961412,48.365663,358.393577,477.262788,118.869211,33.167227


In [ ]:
# ============================================================
# 14. Save outputs
# ============================================================

results_df.to_csv("task1_results_HA_LR_FIXED.csv", index=False)
comparison_df.to_csv("task1_general_vs_incident_comparison_FIXED.csv", index=False)

print("Saved:")
print("- task1_results_HA_LR_FIXED.csv")
print("- task1_general_vs_incident_comparison_FIXED.csv")

# Interpretation guide

Use the tables from Sections 11–13.

For a TraffiDent-style discussion, compare:

- **General Test:** forecasting performance across the ordinary test period.
- **Incident/Post-Incident Test:** forecasting performance when the anchor timestamp is incident/post-incident affected.

Expected TraffiDent-style pattern:

```text
General Test error < Incident/Post-Incident Test error
```

This means post-incident traffic forecasting is harder because incidents create irregular traffic flow patterns.

Recommended metrics to report:

- **MAE**
- **RMSE**
- **Masked MAPE**

Why Masked MAPE?

Traffic flow can be zero or very small. Standard MAPE becomes unstable when actual values are close to zero. Therefore, this notebook reports **Masked MAPE**, which ignores actual flow values less than or equal to the threshold defined in `MAPE_THRESHOLD`.

# Suggested report paragraph

The experiment follows a TraffiDent-style post-incident forecasting setup. Given 24 hours of historical traffic observations, the models predict traffic flow at 1-hour, 3-hour, and 6-hour horizons. The evaluation compares the General Test set with the Incident/Post-Incident Test set. The Historical Average baseline uses the mean of the previous 24 hours, while Linear Regression uses lagged traffic flow, temporal features, current incident indicators, weather variables, and static road attributes available at the anchor timestamp.

If the Incident/Post-Incident Test error is higher than the General Test error, this supports the conclusion that accident or incident events disrupt regular traffic dynamics and make forecasting more difficult. If Linear Regression outperforms Historical Average, this indicates that historical lag patterns and contextual features provide useful predictive information beyond a simple average baseline.